# Data Pipeline
Produces three fixed split CSVs and a class-weights file.
Run once on MacBook. Outputs go to `splits/` and `weights/`.

In [2]:
!pip install -r ../requirements.txt

  Using cached timm-1.0.27-py3-none-any.whl.metadata (40 kB)
  Using cached scikit_multilearn-0.2.0-py3-none-any.whl.metadata (6.0 kB)
  Using cached grad-cam-1.5.5.tar.gz (7.8 MB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached einops-0.8.2-py3-none-any.whl.metadata (13 kB)
  Using cached opencv_python-4.13.0.92-cp37-abi3-macosx_13_0_arm64.whl.metadata (19 kB)
  Using cached jupyter-1.1.1-py2.py3-none-any.whl.metadata (2.0 kB)
  Using cached jupyterlab-4.5.7-py3-none-any.whl.metadata (16 kB)
  Using cached filelock-3.29.0-py3-none-any.whl.metadata (2.0 kB)
  Using cached setuptools-81.0.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached fsspec-2026.4.0-py3-none-any.whl.metadata (10 kB)
  Using cached safetensors-0.7.0-cp38-abi3-macosx_11_0_arm64.whl.metadata 

In [2]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from skmultilearn.model_selection import iterative_train_test_split

# make src/ importable regardless of where the notebook is launched from
PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from src.dataset import LABEL_COLS, remap_uncertain

TRAIN_CSV   = PROJECT_ROOT / 'train.csv'
VALID_CSV   = PROJECT_ROOT / 'valid.csv'
SPLITS_DIR  = PROJECT_ROOT / 'splits'
WEIGHTS_DIR = PROJECT_ROOT / 'weights'
SPLITS_DIR.mkdir(exist_ok=True)
WEIGHTS_DIR.mkdir(exist_ok=True)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('LABEL_COLS:', LABEL_COLS)

PROJECT_ROOT: /Users/davidone/fac/diz/dizertatie_project
LABEL_COLS: ['No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity', 'Lung Lesion', 'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis', 'Pneumothorax', 'Pleural Effusion', 'Pleural Other', 'Fracture', 'Support Devices']


## 1. Load and clean train.csv

In [4]:
df_raw = pd.read_csv(TRAIN_CSV)
print(f'Raw rows: {len(df_raw)}')
print('Columns:', df_raw.columns.tolist())
df_raw.head(3)

Raw rows: 223414
Columns: ['Path', 'Sex', 'Age', 'Frontal/Lateral', 'AP/PA', 'No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity', 'Lung Lesion', 'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis', 'Pneumothorax', 'Pleural Effusion', 'Pleural Other', 'Fracture', 'Support Devices']


,Path,Sex,Age,Frontal/Lateral,AP/PA,No Finding,Enlarged Cardiomediastinum,Cardiomegaly,Lung Opacity,Lung Lesion,Edema,Consolidation,Pneumonia,Atelectasis,Pneumothorax,Pleural Effusion,Pleural Other,Fracture,Support Devices
0,CheXpert-v1.0-small/train/patient00001/study1/...,Female,68,Frontal,AP,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,1.0
1,CheXpert-v1.0-small/train/patient00002/study2/...,Female,87,Frontal,AP,NaN,NaN,-1.0,1.0,NaN,-1.0,-1.0,NaN,-1.0,NaN,-1.0,NaN,1.0,NaN
2,CheXpert-v1.0-small/train/patient00002/study1/...,Female,83,Frontal,AP,NaN,NaN,NaN,1.0,NaN,NaN,-1.0,NaN,NaN,NaN,NaN,NaN,1.0,NaN


In [5]:
# Fix paths: strip 'CheXpert-v1.0-small/' prefix so paths resolve
# from PROJECT_ROOT, e.g. 'train/patient00001/study1/view1_frontal.jpg'
df_raw['Path'] = df_raw['Path'].str.replace('CheXpert-v1.0-small/', '', regex=False)

# Frontal views only — lateral views excluded for Component 1
df = df_raw[df_raw['Frontal/Lateral'] == 'Frontal'].copy()
print(f'Frontal rows: {len(df)}  ({len(df_raw) - len(df)} lateral excluded)')

# Remap uncertain labels (-1) to 0 or 1 per CheXpert paper policy
df = remap_uncertain(df, LABEL_COLS)

# Sanity check: no -1 values should remain
assert (df[LABEL_COLS].values == -1).sum() == 0, 'Uncertain labels remain!'
print('Label value counts (should be 0.0 and 1.0 only):')
print(pd.Series(df[LABEL_COLS].values.ravel()).value_counts())

Frontal rows: 191027  (32387 lateral excluded)
Label value counts (should be 0.0 and 1.0 only):
0.0    2139185
1.0     535193
Name: count, dtype: int64


## 2. Stratified splits

Using `iterative_train_test_split` from scikit-multilearn for multi-label stratification.
Images are the unit of stratification (not patients) — acceptable for dev/colab subsets
since the official validation set is a fully separate patient cohort.

Order of operations:
1. Carve off **5 k** dev subset first (so it is never included in Colab subsets)
2. From the remainder, carve off a **~20 %** Colab validation subset
3. What remains is the **full training set**

In [6]:
N = len(df)
X = np.arange(N).reshape(-1, 1)          # indices as 'features'
y = df[LABEL_COLS].values.astype(int)    # multi-label matrix

# ── Step 1: 5k dev subset ────────────────────────────────────────────────────
dev_ratio = 5_000 / N
X_rest, y_rest, X_dev, y_dev = iterative_train_test_split(X, y, test_size=dev_ratio)
print(f'Dev subset:  {len(X_dev):>6,} images  (target 5,000)')

# ── Step 2: ~20% Colab subset from remainder ─────────────────────────────────
# 20% of the original total, drawn from the non-dev remainder
colab_ratio = (0.20 * N) / len(X_rest)
X_full, y_full, X_colab, y_colab = iterative_train_test_split(
    X_rest, y_rest, test_size=colab_ratio
)
print(f'Colab 20%:   {len(X_colab):>6,} images  (target ~{int(0.20*N):,})')
print(f'Full train:  {len(X_full):>6,} images')

Dev subset:   5,000 images  (target 5,000)
Colab 20%:   38,206 images  (target ~38,205)
Full train:  147,821 images


In [7]:
# Retrieve original dataframe rows using the integer indices stored in X_*
idx_dev   = X_dev.ravel()
idx_colab = X_colab.ravel()
idx_full  = X_full.ravel()

df_dev   = df.iloc[idx_dev].reset_index(drop=True)
df_colab = df.iloc[idx_colab].reset_index(drop=True)
df_full  = df.iloc[idx_full].reset_index(drop=True)

# Verify no overlap
set_dev, set_colab, set_full = set(idx_dev), set(idx_colab), set(idx_full)
assert len(set_dev & set_colab) == 0
assert len(set_dev & set_full)  == 0
assert len(set_colab & set_full) == 0
assert len(set_dev) + len(set_colab) + len(set_full) == N
print('No overlap between splits. Total accounts for all frontal images.')

No overlap between splits. Total accounts for all frontal images.


In [8]:
# Verify label frequency is preserved across splits
print('Positive rate per label (should be similar across splits):\n')
freq = pd.DataFrame({
    'full_train': df_full[LABEL_COLS].mean(),
    'colab_20pct': df_colab[LABEL_COLS].mean(),
    'dev_5k': df_dev[LABEL_COLS].mean(),
    'all': df[LABEL_COLS].mean(),
})
print(freq.round(3).to_string())

Positive rate per label (should be similar across splits):

                            full_train  colab_20pct  dev_5k    all
No Finding                       0.089        0.089   0.089  0.089
Enlarged Cardiomediastinum       0.048        0.048   0.049  0.048
Cardiomegaly                     0.122        0.122   0.123  0.122
Lung Opacity                     0.493        0.493   0.495  0.493
Lung Lesion                      0.037        0.037   0.037  0.037
Edema                            0.322        0.322   0.322  0.322
Consolidation                    0.195        0.198   0.199  0.196
Pneumonia                        0.024        0.024   0.028  0.024
Atelectasis                      0.312        0.312   0.312  0.312
Pneumothorax                     0.093        0.093   0.093  0.093
Pleural Effusion                 0.453        0.453   0.453  0.453
Pleural Other                    0.013        0.013   0.013  0.013
Fracture                         0.039        0.039   0.039  0.039
Su

In [9]:
# Save splits as fixed CSVs — these are committed reference files
df_dev.to_csv(SPLITS_DIR / 'train_dev5k.csv',   index=False)
df_colab.to_csv(SPLITS_DIR / 'train_colab20.csv', index=False)
df_full.to_csv(SPLITS_DIR / 'train_full.csv',    index=False)

# Also clean and save the validation set (strip path prefix, no uncertain
# labels present in valid.csv — but run remap for safety)
df_valid = pd.read_csv(VALID_CSV)
df_valid['Path'] = df_valid['Path'].str.replace('CheXpert-v1.0-small/', '', regex=False)
df_valid = df_valid[df_valid['Frontal/Lateral'] == 'Frontal'].copy()
df_valid = remap_uncertain(df_valid, LABEL_COLS)
df_valid.to_csv(SPLITS_DIR / 'valid_frontal.csv', index=False)

print('Saved:')
for f in sorted(SPLITS_DIR.iterdir()):
    print(f'  {f.name:<25} {pd.read_csv(f).shape}')

Saved:
  train_colab20.csv         (38206, 19)
  train_dev5k.csv           (5000, 19)
  train_full.csv            (147821, 19)
  valid_frontal.csv         (202, 19)


## 3. Class weights

Inverse-frequency weights fitted on the **full training set only**.
Saved and reused across all experiments (baseline, head pruning, token pruning)
so loss scaling is identical across runs.

In [10]:
import json

# Positive frequency per label in the full training set
pos_freq = df_full[LABEL_COLS].mean()   # fraction of positives

# Inverse frequency: rare labels get higher weight
# Clip to avoid division by zero for any label with 0 positives
inv_freq = 1.0 / pos_freq.clip(lower=1e-6)

# Normalise so the mean weight == 1.0 (keeps loss scale comparable
# to unweighted BCE and makes LR tuning more interpretable)
class_weights = (inv_freq / inv_freq.mean()).to_dict()

print('Class weights (higher = rarer label):')
for label, w in sorted(class_weights.items(), key=lambda x: -x[1]):
    print(f'  {label:<35} {w:.4f}  (pos rate {pos_freq[label]:.3f})')

weight_path = WEIGHTS_DIR / 'class_weights.json'
with open(weight_path, 'w') as f:
    json.dump(class_weights, f, indent=2)
print(f'\nSaved to {weight_path}')

Class weights (higher = rarer label):
  Pleural Other                       4.4751  (pos rate 0.013)
  Pneumonia                           2.4064  (pos rate 0.024)
  Lung Lesion                         1.5925  (pos rate 0.037)
  Fracture                            1.5073  (pos rate 0.039)
  Enlarged Cardiomediastinum          1.2205  (pos rate 0.048)
  No Finding                          0.6603  (pos rate 0.089)
  Pneumothorax                        0.6335  (pos rate 0.093)
  Cardiomegaly                        0.4793  (pos rate 0.122)
  Consolidation                       0.3011  (pos rate 0.195)
  Atelectasis                         0.1881  (pos rate 0.312)
  Edema                               0.1823  (pos rate 0.322)
  Pleural Effusion                    0.1296  (pos rate 0.453)
  Lung Opacity                        0.1190  (pos rate 0.493)
  Support Devices                     0.1050  (pos rate 0.559)

Saved to /Users/davidone/fac/diz/dizertatie_project/weights/class_weights.json


## 4. Smoke test

Load one batch through both transform pipelines.
Checks: image shape, label dtype, label values, original size returned correctly.

In [11]:
import torch
from torch.utils.data import DataLoader
from src.dataset import CheXpertDataset, train_transforms, eval_transforms

IMAGE_ROOT = PROJECT_ROOT

for name, tfm in [('train_transforms', train_transforms), ('eval_transforms', eval_transforms)]:
    ds = CheXpertDataset(df_dev.head(16), IMAGE_ROOT, tfm)
    loader = DataLoader(ds, batch_size=4, num_workers=0)
    imgs, labels, pids, sexes, age_grps, orig_sizes = next(iter(loader))

    print(f'\n── {name}')
    print(f'  img shape : {tuple(imgs.shape)}   (expect [4, 3, 224, 224])')
    print(f'  img dtype : {imgs.dtype}           (expect torch.float32)')
    print(f'  img range : [{imgs.min():.3f}, {imgs.max():.3f}]')
    print(f'  labels    : {tuple(labels.shape)}  (expect [4, 14])')
    print(f'  label vals: {labels[0].tolist()}')
    print(f'  patient_id: {list(pids)}')
    print(f'  sex       : {list(sexes)}')
    print(f'  age_grp   : {list(age_grps)}')
    print(f'  orig W    : {orig_sizes[0].tolist()}  (expect variable widths)')

    assert imgs.shape == (4, 3, 224, 224)
    assert labels.shape == (4, 14)
    assert set(labels.unique().tolist()).issubset({0.0, 1.0})

print('\nAll assertions passed.')


── train_transforms
  img shape : (4, 3, 224, 224)   (expect [4, 3, 224, 224])
  img dtype : torch.float32           (expect torch.float32)
  img range : [-2.118, 2.640]
  labels    : (4, 14)  (expect [4, 14])
  label vals: [0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0]
  patient_id: ['patient00002', 'patient00003', 'patient00005', 'patient00005']
  sex       : ['Female', 'Male', 'Male', 'Male']
  age_grp   : ['>60', '40-60', '<40', '<40']
  orig W    : [390, 390, 320, 390]  (expect variable widths)

── eval_transforms
  img shape : (4, 3, 224, 224)   (expect [4, 3, 224, 224])
  img dtype : torch.float32           (expect torch.float32)
  img range : [-2.118, 2.640]
  labels    : (4, 14)  (expect [4, 14])
  label vals: [0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0]
  patient_id: ['patient00002', 'patient00003', 'patient00005', 'patient00005']
  sex       : ['Female', 'Male', 'Male', 'Male']
  age_grp   : ['>60', '40-60', '<40', '<40']
  

## 5. Image integrity scan (5k dev subset)

Verifies every image in the dev subset opens and fully decodes without error.
Catches truncated files or corrupt JPEGs before they crash a training run.
Run the full-dataset version on Colab before the first full training job.

In [3]:
from tqdm import tqdm
from PIL import Image

df_scan = pd.read_csv(SPLITS_DIR / 'train_dev5k.csv')
IMAGE_ROOT = PROJECT_ROOT

failed = []
for _, row in tqdm(df_scan.iterrows(), total=len(df_scan), desc='Scanning dev 5k'):
    path = IMAGE_ROOT / row['Path']
    try:
        img = Image.open(path)
        img.load()   # force full decode — catches truncated files
    except Exception as e:
        failed.append((row['Path'], str(e)))

print(f'\nScanned: {len(df_scan)} images')
if failed:
    print(f'FAILED:  {len(failed)} corrupt or missing files')
    for p, err in failed:
        print(f'  {p}  →  {err}')
else:
    print('All images OK — safe to train on this subset.')

Scanning dev 5k: 100%|██████████| 5000/5000 [00:03<00:00, 1498.96it/s]


Scanned: 5000 images
All images OK — safe to train on this subset.


## 6. Colab path check

Confirms `src/config.py` resolves paths correctly in this environment.
On MacBook `IMAGE_ROOT == PROJECT_ROOT`. On Colab it points to Google Drive.

In [4]:
from src.config import PROJECT_ROOT as CFG_ROOT, IMAGE_ROOT as CFG_IMG, SPLITS_DIR as CFG_SPLITS

print('PROJECT_ROOT :', CFG_ROOT)
print('IMAGE_ROOT   :', CFG_IMG)
print('SPLITS_DIR   :', CFG_SPLITS)
print('splits exist :', CFG_SPLITS.exists())
print('dev csv      :', (CFG_SPLITS / 'train_dev5k.csv').exists())
print('valid csv    :', (CFG_SPLITS / 'valid_frontal.csv').exists())

PROJECT_ROOT : /Users/davidone/fac/diz/dizertatie_project
IMAGE_ROOT   : /Users/davidone/fac/diz/dizertatie_project
SPLITS_DIR   : /Users/davidone/fac/diz/dizertatie_project/splits
splits exist : True
dev csv      : True
valid csv    : True
